# Zomato Restaurants Dataset

## 1.Data Engineering and Dataset Preprocessing

### 1.1 Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


df = pd.read_csv("E:/zomato_restaurants.csv")


print("Raw Data Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Raw Data Shape: (9552, 23)
Columns: ['Unnamed: 0', 'Restaurant ID', 'Restaurant Name', 'Country Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes']


,Unnamed: 0,Restaurant ID,Restaurant Name,Country Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,Average Cost for two,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,0,6317637,Le Petit Souffle,Phillipines,162.0,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",1100.0,Botswana Pula(P),Yes,No,No,No,3.0,4.8,Dark Green,Excellent,314.0
1,1,6304287,Izakaya Kikufuji,Phillipines,162.0,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,1200.0,Botswana Pula(P),Yes,No,No,No,3.0,4.5,Dark Green,Excellent,591.0
2,2,6300002,Heat - Edsa Shangri-La,Phillipines,162.0,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",4000.0,Botswana Pula(P),Yes,No,No,No,4.0,4.4,Green,Very Good,270.0
3,3,6318506,Ooma,Phillipines,162.0,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",1500.0,Botswana Pula(P),No,No,No,No,4.0,4.9,Dark Green,Excellent,365.0
4,4,6314302,Sambo Kojin,Phillipines,162.0,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.58445,"Japanese, Korean",1500.0,Botswana Pula(P),Yes,No,No,No,4.0,4.8,Dark Green,Excellent,229.0


### 2.1 Basic Cleaning (Deduplication + Missing Values)

In [2]:
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

df = df.drop_duplicates()
df = df.drop_duplicates(subset=['Restaurant ID'])

text_cols = ['Restaurant Name', 'Cuisines', 'Locality', 'Locality Verbose', 'Address']
for col in text_cols:
    df[col] = df[col].fillna("Unknown")

num_cols = ['Average Cost for two', 'Aggregate rating', 'Votes']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=['Aggregate rating'])

print("After Cleaning:", df.shape)
print("Missing Values Left:", df.isnull().sum().sum())

After Cleaning: (9552, 22)
Missing Values Left: 14


### 3.1 Data Type Conversion

In [3]:
df['Longitude'] = df['Longitude'].astype(str).str.replace("'", "").str.strip()
df['Latitude'] = df['Latitude'].astype(str).str.replace("'", "").str.strip()

df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')

binary_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map({'Yes': 1, 'No': 0})

print("Data type conversion finished")

Data type conversion finished


### 4.1 Outlier Removal & Feature Engineering

In [4]:
def remove_outlier(data, col):
    Q1, Q3 = data[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    return data[(data[col] >= Q1-1.5*IQR) & (data[col] <= Q3+1.5*IQR)]

df_clean = remove_outlier(df, 'Average Cost for two')
df_clean = remove_outlier(df_clean, 'Aggregate rating')

df_clean['Primary_Cuisine'] = df_clean['Cuisines'].apply(lambda x: x.split(',')[0].strip())

def price_level(price):
    if price < 600: return 'Budget'
    elif 600 <= price <= 1200: return 'Mid-range'
    elif 1200 < price <= 3000: return 'High-end'
    else: return 'Luxury'

df_clean['Price_Level'] = df_clean['Average Cost for two'].apply(price_level)

print("After outlier removal:", df_clean.shape)
print("\nPrice level distribution:")
print(df_clean['Price_Level'].value_counts())

After outlier removal: (6569, 24)

Price level distribution:
Price_Level
Budget       4335
Mid-range    2142
High-end       92
Name: count, dtype: int64


### 5.1 Filter Bangalore & Save Output

In [7]:
# Save only the full cleaned dataset (no city-specific file)
df_clean.to_csv("zomato_clean_all.csv", index=False, encoding='utf-8-sig')

print("\n✅ Data preprocessing completed successfully!")
print("File saved: zomato_clean_all.csv")


✅ Data preprocessing completed successfully!
File saved: zomato_clean_all.csv
